# Calibration scores

In [28]:
import pickle

import numpy as np
import pandas as pd
from sklearn.metrics import (
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

try:
    import config_notebook
except ImportError:
    print("Output will not be deterministic SVG.")

METHOD_LABELS = {
    "sigmoid": "Sigmoid (OvR)",
    "isotonic": "Isotonic (OvR)",
    "temperature": "Temperature",
    "sigmoid_ovo": "Sigmoid (OvO)",
    "isotonic_ovo": "Isotonic (OvO)",
}

In [29]:
y = pd.read_csv("../_temp/labels.csv").to_numpy().reshape(-1)

class_labels = np.sort(pd.unique(y))
label_to_int = {label: idx for idx, label in enumerate(class_labels)}
y = np.array([label_to_int[label] for label in y], dtype=int)

In [30]:
method_names = []
F1_scores = []
NLL_scores = []
Brier_scores = []
Precision_scores = []
Recall_scores = []
ROCAUC_scores = []

for path in [
    "../_temp/CV.sigmoid.pkl",
    "../_temp/CV.isotonic.pkl",
    "../_temp/CV.sigmoid_ovo.pkl",
    "../_temp/CV.isotonic_ovo.pkl",
    "../_temp/CV.temperature.pkl",
]:
    with open(path, "rb") as f:
        data = pickle.load(f)
    method = data["method"]
    outer_splits = data["outer_splits"]
    y_prob_oof = data["y_prob_oof"]

    method_names.append(METHOD_LABELS[method])
    F1 = []
    NLL = []
    Brier = []
    Precision = []
    Recall = []
    ROCAUC = []
    for _, test_idx in outer_splits:
        y_test = y[test_idx]
        y_prob = y_prob_oof[test_idx]
        y_pred = y_prob.argmax(axis=1)
        n_classes = y_prob.shape[1]
        F1.append(f1_score(y_test, y_pred, average="weighted"))
        NLL.append(log_loss(y_test, y_prob))
        Brier.append(
            np.mean(
                [brier_score_loss(y_test == k, y_prob[:, k]) for k in range(n_classes)]
            )
        )
        Precision.append(
            precision_score(y_test, y_pred, average="weighted", zero_division=0)
        )
        Recall.append(recall_score(y_test, y_pred, average="weighted"))
        ROCAUC.append(
            roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted")
        )
    F1_scores.append(np.mean(F1))
    NLL_scores.append(np.mean(NLL))
    Brier_scores.append(np.mean(Brier))
    Precision_scores.append(np.mean(Precision))
    Recall_scores.append(np.mean(Recall))
    ROCAUC_scores.append(np.mean(ROCAUC))

In [31]:
pd.DataFrame(
    {
        "Method": method_names,
        "F1": F1_scores,
        "Precision": Precision_scores,
        "Recall": Recall_scores,
        "ROC-AUC": ROCAUC_scores,
        "NLL": NLL_scores,
        "Brier": Brier_scores,
    }
)

,Method,F1,Precision,Recall,ROC-AUC,NLL,Brier
0,Sigmoid (OvR),0.880073,0.893406,0.883460,0.984534,0.443821,0.028382
1,Isotonic (OvR),0.867753,0.882454,0.870270,0.979876,0.629657,0.027614
2,Sigmoid (OvO),0.855149,0.874714,0.860906,0.980492,1.390234,0.094608
3,Isotonic (OvO),0.877214,0.890642,0.879704,0.979016,1.315940,0.090846
4,Temperature,0.877556,0.890679,0.879721,0.983607,0.379430,0.026131
